In [1]:
import datetime as dt
import time
import random
import importlib
import pandas as pd
from pytrends.request import TrendReq

from equity_data_importers import config

importlib.reload(config)
from equity_data_importers.config import Config

def fetch_google_trends(query: str,
                        start_date: dt.date,
                        end_date: dt.date,
                        geo: str = None,
                        window_days: int = 200
                       ) -> pd.DataFrame:
    """
    Pulls Google Trends data in `window_days`-sized chunks
    from start_date → end_date (inclusive). Returns a DataFrame
    indexed by Date with one column, 'trends_score'.
    """
    delta = dt.timedelta(days=window_days)
    current_start = start_date
    collected = []

    user_agents = [
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
        "Mozilla/5.0 (X11; Linux x86_64)",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)"
    ]

    print(f"Pobieram Google Trends dla: {query}")
    while current_start <= end_date:
        current_end = min(current_start + delta, end_date)
        tf = f"{current_start.strftime('%Y-%m-%d')} {current_end.strftime('%Y-%m-%d')}"

        try:
            headers = {"User-Agent": random.choice(user_agents)}
            pt = TrendReq(
                hl="en-US",
                tz=360,
                retries=3,
                backoff_factor=0.5,
                requests_args={"headers": headers}
            )
            pt.build_payload([query], timeframe=tf, geo=geo)
            df = pt.interest_over_time().drop(columns="isPartial", errors="ignore")

            if df.empty:
                print(f"Brak danych dla przedziału {tf}")
            else:
                collected.append(df)
                print(f"✅ Trendy {tf}: OK ({len(df)} wierszy)")

        except Exception as e:
            print(f"Błąd Trends ({tf}): {e}")

        # advance
        current_start = current_end + dt.timedelta(days=1)
        wait = random.randint(60, 90)
        print(f"Pauza {wait} sek...")
        time.sleep(wait)

    if not collected:
        print("⚠️ Google Trends puste na całym przedziale.")
        return pd.DataFrame(columns=["trends_score"],
                             index=pd.date_range(start_date, end_date, freq="D"))

    # concatenate, drop any overlapping dates, rename column
    full = pd.concat(collected).sort_index()
    full = full[~full.index.duplicated()]
    full.rename(columns={query: "trends_score"}, inplace=True)
    return full[["trends_score"]]


def main():
    df = fetch_google_trends(
        query=Config.COMPANY_NAME,
        start_date=Config.START_DATE,
        end_date=Config.END_DATE,
        geo=Config.GEO,
        window_days=200
    )

    out_file = f"../data/equity_data/google_trends_data.csv"
    df.to_csv(out_file, index=True)
    print(f"Dane zapisane do pliku: {out_file}")


if __name__ == "__main__":
    main()


🔍 Pobieram Google Trends dla: Tesla
✅ Trendy 2023-01-01 2023-07-20: OK (201 wierszy)
🕒 Pauza 72 sek...
✅ Trendy 2023-07-21 2024-02-06: OK (201 wierszy)
🕒 Pauza 84 sek...
✅ Trendy 2024-02-07 2024-08-25: OK (201 wierszy)
🕒 Pauza 88 sek...
✅ Trendy 2024-08-26 2025-03-14: OK (201 wierszy)
🕒 Pauza 86 sek...
✅ Trendy 2025-03-15 2025-10-01: OK (201 wierszy)
🕒 Pauza 76 sek...
✅ Trendy 2025-10-02 2026-01-01: OK (92 wierszy)
🕒 Pauza 73 sek...
💾 Dane zapisane do pliku: data/google_trends_data.csv
